In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/OE/Kerendia_OE_Inference/00_Helper_Notebooks/00_connection_inference"

In [0]:
suppression1 = Get_Data_Snowflakes("""select distinct BH_ID,AMA_DO_NOT_CONTACT,KAISER,PDRP_OPTOUT 
                    from PHCDW.PHCDW_STG.MDM_STG_HCP_PRFL 
                    where (AMA_DO_NOT_CONTACT ='Y' or KAISER ='Y') and BH_ID is not null""")
suppression2 = Get_Data_Snowflakes("""select distinct BH_ID, ADMINISTRATIVE_AREA 
                                   from PHCDW.PHCDW_STG.MDM_STG_HCP_ADDR 
                                   where administrative_area in ('CO','VT') and address_ordinal ='1'""")


Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


[REDACTED]

[REDACTED]

[REDACTED]

[REDACTED]



In [0]:
from datetime import datetime

# Delta output base path
output_base_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/Inference/Promotional_data_ingestion/"

# Current date as YYYYMMDD for naming
current_date_str = datetime.today().strftime('%Y%m%d')
print(f"Current date: {current_date_str}")
print(f"Filter: DATE_ID >= 24 months back from current date (includes future dates)")

Current date: 20261105
Filter: DATE_ID >= 24 months back from current date (includes future dates)


In [0]:
# Table 1: FACT_CALL joined with MAP_CUSTOMER_XREF
# Filter: 24 months back from current date, includes future dates

df_call = Get_Data_Snowflakes("""
    SELECT DISTINCT 
        B.BAYER_CUST_ID AS BHID,
        CASE 
            WHEN UPPER(A.CALL_REC_TYP_NM) LIKE '%BPH_FACE2FACE_SIG%' OR UPPER(A.CALL_REC_TYP_NM) LIKE 'FACE2FACE_SIG%' THEN 'F2F'
            WHEN UPPER(A.CALL_REC_TYP_NM) LIKE '%BPH_NON_FACE2FACE%' THEN 'PHONE'
            WHEN UPPER(A.CALL_REC_TYP_NM) IN ('BPH_GROUP_CALL_SIG', 'GROUP CALL SIGNED', 'EVENT_VOD') THEN 'GROUP_CALL'
            WHEN UPPER(A.CALL_REC_TYP_NM) LIKE '%BPH_REMOTE%' THEN 'REMOTE_CALL'
            ELSE A.CALL_REC_TYP_NM 
        END AS CHNNL_NM,
        A.DATE_ID,
        A.CALL_ID,
        A.CALL_TYP,
        A.DETL_PROD
    FROM CPH_DB_PROD.MODEL_V2.FACT_CALL A
    LEFT JOIN (
        SELECT SOURCE_CUST_ID, BAYER_CUST_ID 
        FROM CPH_DB_PROD.MODEL_V2.MAP_CUSTOMER_XREF
    ) B
    ON A.SOURCE_CUST_ID = B.SOURCE_CUST_ID
    WHERE UPPER(A.STAT) = 'SUBMITTED_VOD'
        AND UPPER(A.DETL_PROD) = 'KERENDIA'
        AND B.BAYER_CUST_ID NOT LIKE '%BHO%'
        AND B.BAYER_CUST_ID IS NOT NULL
        AND A.DATE_ID >= TO_CHAR(DATEADD('month', -24, CURRENT_DATE()), 'YYYYMMDD')
""")

print(f"df_call rows: {df_call.count()}")
df_call.display()

df_call rows: 960483


BHID,CHNNL_NM,DATE_ID,CALL_ID,CALL_TYP,DETL_PROD
BH15334269,F2F,20251217,V630000000M2868,Group Detail,Kerendia
BH10139904,F2F,20250915,V6300000007W285,Detail Only,Kerendia
BH11304446,F2F,20250908,V6300000006C536,Group Detail,Kerendia
BH11304446,F2F,20251006,V630000000BK108,Group Detail,Kerendia
BH13168379,F2F,20260515,V63000000198475,Group Detail,Kerendia
BH33821421,F2F,20260724,V630000001MG299,Group Detail,Kerendia
BH33300355,F2F,20260302,V630000000V9545,Detail Only,Kerendia
BH10128454,F2F,20250430,a04Vt00000IY3xBIAT,Group Detail,Kerendia
BH10574831,F2F,20250709,a04Vt00000N3OtTIAV,Detail Only,Kerendia
BH23418483,F2F,20260914,V630000001WP730,Group Detail,Kerendia


In [0]:
# Table 2: Non-Personal Promotions
# Filter: 24 months back from current date, includes future dates

npp_columns = [
    'CUST_ID', 'PROD_ID', 'LITCODE', 'DATE_ID', 'CHNL_ID', 'MTR_NM', 'VEND_NM',
    'CHNL_NM', 'RPT_CHNL', 'CPGN_NM', 'TACT_TYP', 'HCP_SEG_1', 'CHNL_TYP',
    'CPGN_AUDS', 'EMAIL_SUBJ_LN', 'CRTV_NM', 'TACT_NM', 'PRMRY_KEY_MSG',
    'COUNTRY_NM', 'HCP_PRMRY_SPCL', 'HCP_TYP', 'HCP_ID_TYP', 'CF_NAME',
    'CF_INDICATION', 'LINK_URL', 'TKN_ID', 'DEVC', 'VEND_PROD', 'BAYER_HCP_ID',
    'HCP_PRMRY_CITY', 'HCP_PRMRY_STATE', 'HCP_PRMRY_ZIP', 'HCP_PRMRY_STATE_LIC_NBR',
    'HCP_PAR_SPCL_GRP', 'PROD_ID_TYP', 'BAYER_PROD_ID', 'CNTT_NM', 'UNIQ_NPP_ID',
    'TRGT_FLAG', 'BUS_UNIT', 'PROD_BRAND_NM'
]

cols_str = ", ".join(npp_columns)

df_npp = Get_Data_Snowflakes(f"""
    SELECT {cols_str}
    FROM CPH_DB_PROD.ANALYTICS_V2.ANLT_BASE_FACT_NONPERSONAL_PROMOTIONS
    WHERE DATE_ID >= TO_CHAR(DATEADD('month', -24, CURRENT_DATE()), 'YYYYMMDD')
""")

print(f"df_npp rows: {df_npp.count()}")
df_npp.display()

df_npp rows: 409707392


CUST_ID,PROD_ID,LITCODE,DATE_ID,CHNL_ID,MTR_NM,VEND_NM,CHNL_NM,RPT_CHNL,CPGN_NM,TACT_TYP,HCP_SEG_1,CHNL_TYP,CPGN_AUDS,EMAIL_SUBJ_LN,CRTV_NM,TACT_NM,PRMRY_KEY_MSG,COUNTRY_NM,HCP_PRMRY_SPCL,HCP_TYP,HCP_ID_TYP,CF_NAME,CF_INDICATION,LINK_URL,TKN_ID,DEVC,VEND_PROD,BAYER_HCP_ID,HCP_PRMRY_CITY,HCP_PRMRY_STATE,HCP_PRMRY_ZIP,HCP_PRMRY_STATE_LIC_NBR,HCP_PAR_SPCL_GRP,PROD_ID_TYP,BAYER_PROD_ID,CNTT_NM,UNIQ_NPP_ID,TRGT_FLAG,BUS_UNIT,PROD_BRAND_NM
3517de5786016cd46090e518e067f8db482bdf322e9f2a4bfcec0360b4b5d1c4,e1977d75b45e0def0718d38f9a3e463204603d826c89fbd00776aaeffe262a8d,null,20241216,IDETIFIED_WEB_ACTIVITY,page_view,DMD_AIM,null,Website Visit,,null,,Website Visit,HCP,,null,null,,USA,null,null,NON-MASTERED,null,null,www.nubeqahcp.com/,null,null,,-1,,null,,,Non Mastered,NON-MASTERED,-1,BRN_X_ALL_PHR_X_X,1c5809cf84c836b7ff71d9181698ed47fe1af2a3ddaa2fb6d9c8475ec446b25a,0,ONC,NUBEQA
3517de5786016cd46090e518e067f8db482bdf322e9f2a4bfcec0360b4b5d1c4,a79d37090b11dc0489d396283f1d0062aea66fa336592ea3f873467f5e974ffd,null,20260114,IDETIFIED_WEB_ACTIVITY,page_view,DMD_AIM,null,Website Visit,,null,,Website Visit,HCP,,null,null,,USA,null,null,NON-MASTERED,null,null,www.nubeqa-us.com/about-prostate-cancer/nmcrpc,null,null,,-1,,null,,,Non Mastered,NON-MASTERED,-1,BRN_X_ALL_PHR_X_X,23f4919fa5c1efa2ec300d75afc2aea21a5fc5060863ae6a86c561b0c5853c32,0,null,NUBEQA-V4
BH12999350,1f63dc7b693c1b455d57c6ef8183766754132ee654d6e8127f56661a0bde9849,null,20260401,BANNER,Impressions,PULSEPOINT,BANNER,Banner,Admps-2026_HCP_plsp,Programatic,Adempas HCP_TL_Other_4.15.25,Paid Media,HCP,null,ADMPS_HCP_display_plsp_OE_AWA_TL_CTEPH_ImLnk_300x250_STAND,null,null,USA,PUD,Prescriber,MASTERED,null,null,,,Phone,null,BH12999350,Lafayette,LA,70508,MD.206899,PCP,NON-MASTERED,-1,ADMPS_HCP_display_plsp_OE_AWA_TL_CTEPH_ImLnk_300x250_STAND,645503fe50e0c37591187e6ab61731d3402bceceffb5461634d9abbf5a26104a,1,CVR,ADEMPAS
BH11231166,17ffb98cec941c6decd92f3c9fe4188f21d43a717d9fef9baaf4b9d00c21c6ad,null,20250716,BANNER,Impressions,PULSEPOINT,BANNER,Banner,Ker-2025_HCP_plsp,Programatic,Kerendia HCP_TL_Awareness_4.14.25,Paid Media,HCP,null,KER_HCP_display_plsp_OE_AWA_TL-Awareness_MOD_ImLnk_728x90_STAND,null,null,USA,CD,Prescriber,MASTERED,null,null,,,Phone,null,BH11231166,Tucson,AZ,85719,52334,Cardiology,NON-MASTERED,-1,KER_HCP_display_plsp_OE_AWA_TL-Awareness_MOD_ImLnk_728x90_STAND,3838e6979727887aecc8f2baf0e64cc638f95158ebd3125f3098ac02360abd5f,0,CVR,KERENDIA
BH11467196,17ffb98cec941c6decd92f3c9fe4188f21d43a717d9fef9baaf4b9d00c21c6ad,null,20260611,BANNER,Impressions,PULSEPOINT,BANNER,Banner,KER-2026_HCP_plsp,Programatic,Kerendia HCP_NBA_API Upload,Paid Media,HCP,null,KER_HCP_display_plsp_OE_AWA_TL-NBA_DualIndication_ImLnk_300x50_STAND,null,null,USA,FM,Prescriber,MASTERED,null,null,,,Phone,null,BH11467196,Olean,NY,14760,60271453,PCP,NON-MASTERED,-1,KER_HCP_display_plsp_OE_AWA_TL-NBA_DualIndication_ImLnk_300x50_STAND,092f703403710486989d7fdd851206a3581c284c6cb1b54facb5e2b4bb660177,1,CVR,KERENDIA
BH11940704,1f63dc7b693c1b455d57c6ef8183766754132ee654d6e8127f56661a0bde9849,null,20260225,BANNER,Impressions,PULSEPOINT,BANNER,Banner,Admps-2026_HCP_plsp,Programatic,Adempas HCP_TL_Other_4.15.25,Paid Media,HCP,null,ADMPS_HCP_display_plsp_OE_AWA_TL_Subgroup_ImLnk_300x250_STAND,null,null,USA,CD,Prescriber,MASTERED,null,null,,,Phone,null,BH11940704,Milwaukee,WI,53215,7109420,Cardiology,NON-MASTERED,-1,ADMPS_HCP_display_plsp_OE_AWA_TL_Subgroup_ImLnk_300x250_STAND,829e3ba202b105f26086c8d2ae84f0bd32ec67526a88c5c7faacde1b9cf59547,1,CVR,ADEMPAS
BH10470282,17ffb98cec941c6decd92f3c9fe4188f21d43a717d9fef9baaf4b9d00c21c6ad,null,20250309,BANNER,Impressions,PULSEPOINT,BANNER,Banner,Ker-2025_HCP_plsp,Programatic,Kerendia HCP_TL_Adoption_2.28.25,Paid Media,HCP,null,KER_HCP_display_plsp_OE_AWA_TL-Adoption_PatAccess1_ImLnk_300x50_STAND,null,null,USA,CD,Prescriber,MASTERED,null,null,,,Phone,null,BH10470282,Cincinnati,OH,45219,35096425,Cardiology,NON-MASTERED,-1,KER_HCP_display_plsp_OE_AWA_TL-Adoption_PatAccess1_ImLnk_300x

In [0]:
# Table 3: Personal Promotion Events
# Filter: 24 months back from current date, includes future dates

evnt_columns = [
    'EVNT_ID', 'PROD_ID', 'DATE_ID', 'CUST_HCP_ID', 'EVNT_CAT',
    'EVNT_TYP', 'EVNT_SUB_TYP', 'ATND', 'EVNT_FMT', 'EVNT_TPIC_NM', 'PROD_BRAND_NM'
]

cols_str = ", ".join(evnt_columns)

df_evnt = Get_Data_Snowflakes(f"""
    SELECT {cols_str}
    FROM CPH_DB_PROD.ANALYTICS_V2.ANLT_BASE_FACT_EVNT_PRSNL_PROM
    WHERE DATE_ID >= TO_CHAR(DATEADD('month', -24, CURRENT_DATE()), 'YYYYMMDD')
""")

print(f"df_evnt rows: {df_evnt.count()}")
df_evnt.display()

df_evnt rows: 69668


EVNT_ID,PROD_ID,DATE_ID,CUST_HCP_ID,EVNT_CAT,EVNT_TYP,EVNT_SUB_TYP,ATND,EVNT_FMT,EVNT_TPIC_NM,PROD_BRAND_NM
V7R000000043010,P50419-395,20260717,null,NON-SPEAKER PROGRAMS,Sponsorship,null,null,Face-to-Face,Display/Exhibit,NUBEQA
V7R00000004D013,P50419-540,20260514,BH10043659,SPEAKER PROGRAMS,Speaker Program,null,null,Face-to-Face,KERENDIA: At the heart of reducing CV risk in adult patients with CKD and T2D (PP-KER-US-2850-2),KERENDIA
V7R00000001C011,P50419-540,20251113,BH48810175,SPEAKER PROGRAMS,Speaker Program,null,Attended,Face-to-Face,KERENDIA: At the heart of reducing CV risk in adult patients with CKD and T2D (PP-KER-US-2850-2),KERENDIA
a2vVt000004TQ4rIAG,P50419-540,20250515,BH22612812,SPEAKER PROGRAMS,Speaker Program,null,null,Face-to-Face,(Condensed) KERENDIA: At the heart of reducing CV risks in adult patients with CKD and T2D PP-KER-US-2530-3,KERENDIA
V7R000000056015,P50419-540,20260618,BH11111963,NON-SPEAKER PROGRAMS,Scientific Presentation,null,Attended,Remote,null,KERENDIA
a2vVt000004ou7xIAA,P50419-540,20250703,BH12879101,SPEAKER PROGRAMS,Speaker Program,null,Attended,Face-to-Face,**KERENDIA: At the heart of reducing CV risks in adult patients with CKD and T2D PP-KER-US-2529-3,KERENDIA
V7R00000007E044,VALZ06G6O7O5PXF,20261117,null,NON-SPEAKER PROGRAMS,medical_event__c,null,null,Face-to-Face,Asundexian {Med} (US),ASUNDEXIAN [SUNHARBO]
a2vVt000004gIKnIAM,P50419-395,20250809,null,NON-SPEAKER PROGRAMS,Sponsorship,null,null,Face-to-Face,Display/Exhibit,NUBEQA
V7R00000001G026,P50419-540,20251120,null,SPEAKER PROGRAMS,Speaker Program,null,null,Face-to-Face,**The next beat: KERENDIA (finerenone) for your patients with HF with LVEF ≥40% (PP-KER-US-2848-2),KERENDIA
V7R00000000C007,P50419-540,20250918,BH12910535,SPEAKER PROGRAMS,Speaker Program,null,null,Face-to-Face,**The next beat: KERENDIA (finerenone) for your patients with HF with LVEF ≥40% (PP-KER-US-2848-2),KERENDIA


In [0]:
# Save all 3 filtered DataFrames as UC managed tables

TABLE_CATALOG = "`ph-com-ai-us-prod-catalog-2553213839599380`"
TABLE_SCHEMA = "kerendia_oe_journey"

# Table 1: FACT_CALL
df_call.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_CATALOG}.{TABLE_SCHEMA}.oe_kerendia_fact_call_inference")
print("Saved: oe_kerendia_fact_call_inference")

# Table 2: Non-Personal Promotions
df_npp.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_CATALOG}.{TABLE_SCHEMA}.oe_kerendia_fact_npp_inference")
print("Saved: oe_kerendia_fact_npp_inference")

# Table 3: Personal Promotion Events
df_evnt.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{TABLE_CATALOG}.{TABLE_SCHEMA}.oe_kerendia_fact_evnt_prsnl_prom_inference")
print("Saved: oe_kerendia_fact_evnt_prsnl_prom_inference")

Saved: oe_kerendia_fact_call_inference
Saved: oe_kerendia_fact_npp_inference
Saved: oe_kerendia_fact_evnt_prsnl_prom_inference


## 🎯 Analysis 6: Monthly HCP Density + Waterfall — TARGET LIST HCPs Only (All 3 Tables)

### What are we doing?
We repeat the same two analyses — **monthly HCP density** and **full pipeline waterfall** — but this time **restricted to only the target list HCPs** (`modelling_flag = 'Yes'` from `13_5_oe_modelling_universe_final`).

### Why is this different from previous analyses?
| Previous analyses (A1–A5) | This analysis (A6) |
|---|---|
| Counted ALL HCPs in the raw tables | Counts only HCPs on the target list |
| Showed sparsity in the broad Snowflake population | Shows sparsity specifically for HCPs we need to score |
| Universe join was a diagnostic step | Target list is applied FIRST before any counting |

### What does this tell us?
If the sparsity pattern persists even after filtering to the target list, it means **the HCPs we need to score are themselves being underserved** by the source data in 2026 — not just the broader population.

In [0]:
from pyspark.sql import functions as F

df_npp.groupBy("VEND_NM").agg(F.max("DATE_ID").alias("MAX_DATE")).orderBy("MAX_DATE", ascending=False).display()

VEND_NM,MAX_DATE
SFMC,20261104
DOCEREE,20261102
DMD,20261102
OPENEVIDENCE,20261101
VUMEDI,20261101
MD_CALC,20261031
PULSEPOINT,20261030
PEERDIRECT,20261029
iRep Email,20261028
DOXIMITY,20261027
